# Part 1 — Run SPINE on files (full and lite output)

In this notebook you will:
1. look inside a production config, to see what full vs lite changes;
2. run the SPINE **full chain** on a small **generic** file and a small **ProtoDUNE-SP** file;
3. write each one as a **full** and as a **lite** `.h5` file;
4. compare the four output files.

**Session needed:** Jupyter on partition `ampere` with **1 GPU** (see Part 0, Path A).

> **Terminal equivalent** (from `compute_shell.sh`; full explanation in [`README.md`](README.md)):
> ```bash
> cd $WORKDIR
> spine_container.sh spine -c infer/generic/full_chain_240805.yaml \
>     -s $TUTORIAL_DATA/generic_small.root --output-dir $WORKDIR/out --output-suffix full --log-dir $WORKDIR/logs
> spine_container.sh spine -c $SPINE_TUTORIAL/configs/generic_full_chain_lite.yaml \
>     -s $TUTORIAL_DATA/generic_small.root --output-dir $WORKDIR/out --output-suffix lite --log-dir $WORKDIR/logs
> ```

## 0. Setup

In [ ]:
import sys; sys.path.insert(0, "..")
import os
import tutorial_env as te
env = te.setup()

OUT = os.path.join(env["WORKDIR"], "out")       # all outputs of this notebook go here
LOGS = os.path.join(env["WORKDIR"], "logs")
os.makedirs(OUT, exist_ok=True)

# On a GPU node use it; otherwise fall back to CPU (slow, generic file only)
GPU = te.has_gpu()
WORLD = "" if GPU else "--world-size 0"
print("GPU available:", GPU)

Get the two input files (downloaded once, then reused):

In [ ]:
generic_root = te.fetch("generic_small.root")
pdsp_root = te.fetch("protodune-sp_small.root")

## 1. What changes between full and lite?

A **config** tells SPINE what to read, which model to run and what to write. spine-prod's
production configs are built from several pieces. `spine-config dump` shows the final
result with all pieces merged. Here we only look at the `writer` block, which controls the output file:

In [ ]:
te.sh("spine-config dump $SPINE_PROD_BASEDIR/config/infer/generic/full_chain_240805.yaml | grep -A 16 '  writer:'")

In [ ]:
te.sh("spine-config dump $SPINE_TUTORIAL/configs/generic_full_chain_lite.yaml | grep -A 12 '  writer:'")

**What you should see:** the full config's `keys` list includes `points`, `depositions`,
`points_label`, `depositions_label` (the voxels). The lite one has `lite: true`, and its list
keeps only `run_info`, `meta` and the four particle/interaction collections.

The lite config is just two lines of `include` (open [`../configs/generic_full_chain_lite.yaml`](../configs/generic_full_chain_lite.yaml)):
the production bundle, plus spine-prod's `lite` modifier.

## 2. Generic detector: full output

`te.sh(...)` runs a shell command and prints it first (the `$ ...` line). You could paste that
line into a terminal and get the same result.

The first time, SPINE downloads the trained weights (115 MB) into `$SPINE_CACHE_DIR`. After that it reuses them.

In [ ]:
te.sh(f"""spine -c infer/generic/full_chain_240805.yaml \
          -s {generic_root} {WORLD} \
          --output-dir {OUT} --output-suffix full --log-dir {LOGS}""")

**What you should see:**
- `Will load 1 file(s)` and the number of entries.
- A table with timing for each batch (`Time (inference)`, GPU memory, …).
- No `Traceback` at the end.

The output is `generic_small_full.h5`, named `<input name>_<suffix>.h5`.

## 3. Generic detector: lite output

In [ ]:
te.sh(f"""spine -c $SPINE_TUTORIAL/configs/generic_full_chain_lite.yaml \
          -s {generic_root} {WORLD} \
          --output-dir {OUT} --output-suffix lite --log-dir {LOGS}""")

## 4. ProtoDUNE-SP: full and lite (first 10 events)

ProtoDUNE-SP events are much bigger than generic ones, so `-n 10` processes only the first
10 entries. This step needs a GPU.

In [ ]:
if GPU:
    te.sh(f"""spine -c infer/protodune-sp/full_chain_260906.yaml \
              -s {pdsp_root} -n 10 \
              --output-dir {OUT} --output-suffix full --log-dir {LOGS}""")
    te.sh(f"""spine -c $SPINE_TUTORIAL/configs/protodune-sp_full_chain_lite.yaml \
              -s {pdsp_root} -n 10 \
              --output-dir {OUT} --output-suffix lite --log-dir {LOGS}""")
else:
    print("Skipping ProtoDUNE-SP: no GPU in this session.")

## 5. Compare the four files

`h5py` opens HDF5 files like a dictionary of tables. Each top-level key is one *data product*.

In [ ]:
import glob
import h5py
import pandas as pd

rows = []
for path in sorted(glob.glob(os.path.join(OUT, "*_full.h5")) + glob.glob(os.path.join(OUT, "*_lite.h5"))):
    with h5py.File(path, "r") as f:
        rows.append({
            "file": os.path.basename(path),
            "size [MB]": round(os.path.getsize(path) / 1e6, 2),
            "entries": len(f["events"]),
            "has voxels (points)": "points" in f,
            "products": ", ".join(k for k in f.keys() if k not in ("events", "index", "info")),
        })
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(rows)

**What you should see:** each `_lite` file is several times smaller than its `_full`
partner and has no `points`/`depositions`.

Every SPINE file also remembers **how it was made**: the SPINE version and the full
config are stored in the `info` attributes.

In [ ]:
with h5py.File(os.path.join(OUT, "generic_small_lite.h5"), "r") as f:
    for key, value in f["info"].attrs.items():
        text = str(value)
        print(f"{key:<16}", text[:150] + (" ..." if len(text) > 150 else ""))

## 6. A first look at what's inside (preview of Part 2)

`HDF5Reader` turns the stored tables back into Python objects. Here are the reconstructed
particles of the first event in the lite file:

In [ ]:
from spine.io.read import HDF5Reader
from spine.constants import PID_LABELS, SHAPE_LABELS

reader = HDF5Reader(file_keys=os.path.join(OUT, "generic_small_lite.h5"))
event = reader[0]
print(f"{len(reader)} entries; entry 0 has {len(event['reco_particles'])} reco particles\n")
for p in event["reco_particles"]:
    print(f"particle {p.id:>2}: {SHAPE_LABELS[p.shape]:<7} {PID_LABELS[p.pid]:<9} "
          f"primary={str(bool(p.is_primary)):<5} KE={p.ke:8.1f} MeV")

## 7. (Optional) The same thing through spine-prod: `submit.py -I`

`submit.py` is how productions are run. With `-I` it runs right here instead of submitting
a batch job, so you can check the exact production setup on one file. `infer/generic/latest`
means "the newest generic configuration", and `--apply-mods lite` adds the lite modifier for you.

In [ ]:
os.chdir(env["WORKDIR"])            # submit.py creates its runs/ folder in the current directory
te.sh(f"""$SPINE_PROD_BASEDIR/submit.py -I \
          --config infer/generic/latest --apply-mods lite \
          --source {generic_root} --bind-paths /sdf \
          --output-suffix spine_lite {WORLD}""")
te.sh("ls -d runs/*interactive* && ls runs/*interactive*/output | tail -3")

## Summary

| You did | Command |
|---|---|
| inspected a config | `spine-config dump <config>` |
| ran SPINE, full output | `spine -c infer/<det>/full_chain_<date>.yaml -s in.root --output-dir … --output-suffix full` |
| ran SPINE, lite output | same with a lite config, or `submit.py --apply-mods lite` |
| tested the production recipe | `submit.py -I --config infer/<det>/latest …` |

To process **many files as batch jobs**, follow section 5 of [`README.md`](README.md#5-way-c--batch-jobs-with-slurm),
which must be done from an `ssh` terminal on iana.

Next: **Part 2** — [`../02_analysis/02a_open_larcv_files.ipynb`](../02_analysis/02a_open_larcv_files.ipynb) and
[`../02_analysis/02b_analyze_spine_output.ipynb`](../02_analysis/02b_analyze_spine_output.ipynb).